# Day 4 — Big-O by Counting Steps

No new problems today. This notebook is a lab: tiny loops with a **step counter** inside, so I can *see* how the number of operations grows with `n` instead of trusting formulas.

The one habit to build: before running any cell, **predict the counter value**. Then run and check. If my prediction matches, I understand the complexity.

At the end: my self-audit checklist and the table I must fill for every problem solved so far (Day-1 digit problems + patterns 1–15).

## 1. O(n) vs O(n²) — same n, very different step counts

Both functions below take the same `n = 8`. Each puts `steps += 1` inside the loop body, so the counter tells the truth about how much work happened.

**What to observe / predicted output:**

- Single loop (chai for every guest): body runs once per guest → predict **8**.
- Nested loop (every wedding guest greets every guest): inner body runs n times for *each* outer pass → predict **8 × 8 = 64**.

```
single loop, n=8  -> steps = 8
nested loop, n=8  -> steps = 64
```

Then imagine n = 1000: the single loop does 1,000 steps, the nested one does 1,000,000. Doubling n doubles the first counter but **quadruples** the second.

In [ ]:
def single_loop(n):          # O(n) shape
    steps = 0
    for i in range(n):
        steps += 1           # one step per guest
    return steps

def nested_loop(n):          # O(n^2) shape
    steps = 0
    for i in range(n):
        for j in range(n):
            steps += 1       # every guest x every guest
    return steps

n = 8
print("single loop, n=8  -> steps =", single_loop(n))
print("nested loop, n=8  -> steps =", nested_loop(n))

## 2. The halving loop — why it is O(log n)

Phone book method: open the middle, throw away half, repeat. In code, that is a loop doing `n = n // 2` each pass.

"log₂(n)" simply means: *how many times can I halve n before it reaches 1?*

**What to observe / predicted output:** `n = 1000` should take about **10** halvings, because 2¹⁰ = 1024 ≈ 1000. (Exact count here: 9 — the chain is 1000 → 500 → 250 → 125 → 62 → 31 → 15 → 7 → 3 → 1.) A million should take about 20 (exact: 19).

```
n = 1000     -> steps = 9    (~ log2 1000 ~ 10)
n = 1000000  -> steps = 19   (~ log2 1000000 ~ 20)
```

The input grew 1000×, the work grew 2×. That is the magic of logarithmic growth. (A digit-stripping loop with `n //= 10` is the same idea in base 10 — one step per digit.)

In [ ]:
def halving_steps(n, show=False):     # O(log n) shape
    steps = 0
    while n > 1:
        n = n // 2                    # throw away half the phone book
        steps += 1
        if show:
            print("  after halving:", n)
    return steps

print("n = 1000")
print("steps =", halving_steps(1000, show=True))
print()
print("n = 1000000 -> steps =", halving_steps(1000000))

## 3. Linear search vs binary search — the counter settles it

Same sorted list of 1,000 numbers, same target: the **last** one (worst case for linear search). Linear search checks one by one. Binary search is the phone book method in code: look at the middle, throw away the wrong half, repeat.

**What to observe / predicted output:** linear should need about **1000** checks; binary about **10**, because 2¹⁰ ≈ 1000.

```
linear search, n=1000 -> steps = 1000
binary search, n=1000 -> steps = 10
```

Same job, ~100× fewer steps. But remember the fine print: binary search only works because the list is **sorted**. On unsorted data it is not slow — it is *wrong*.

In [ ]:
def linear_search_steps(arr, target):      # O(n) shape
    steps = 0
    for i in range(len(arr)):
        steps += 1                         # one check per item
        if arr[i] == target:
            break
    return steps

def binary_search_steps(arr, target):      # O(log n) shape — needs SORTED data
    steps = 0
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        steps += 1                         # one check per halving
        mid = (lo + hi) // 2
        if arr[mid] == target:
            break
        elif arr[mid] < target:
            lo = mid + 1                   # throw away left half
        else:
            hi = mid - 1                   # throw away right half
    return steps

arr = list(range(1, 1001))                 # sorted list: 1..1000
target = 1000                              # last item = worst case for linear

print("linear search, n=1000 -> steps =", linear_search_steps(arr, target))
print("binary search, n=1000 -> steps =", binary_search_steps(arr, target))

## 4. Sequential loops ADD, nested loops MULTIPLY

Two loops written one **after** the other are nothing like one loop **inside** the other. Same `n = 8` for both.

**What to observe / predicted output:**

- Sequential: n steps, then n more → **8 + 8 = 16** → O(n + n) = O(2n) = **O(n)** after dropping the constant.
- Nested: n steps for each of n passes → **8 × 8 = 64** → **O(n²)**.
- Triangle (inner loop runs `i` times): 0+1+2+...+7 = n(n−1)/2 = **28** → still **O(n²)** shape, just with a ½ constant that Big-O drops.

```
sequential, n=8 -> steps = 16   (add:      n + n)
nested,     n=8 -> steps = 64   (multiply: n * n)
triangle,   n=8 -> steps = 28   (1+2+...+(n-1) = n(n-1)/2)
```

The triangle result is exactly why my star-triangle patterns are O(n²) even though the inner loop is "shorter".

In [ ]:
def sequential(n):               # loop AFTER loop -> add
    steps = 0
    for i in range(n):
        steps += 1
    for j in range(n):
        steps += 1
    return steps                 # n + n = 2n -> O(n)

def nested(n):                   # loop INSIDE loop -> multiply
    steps = 0
    for i in range(n):
        for j in range(n):
            steps += 1
    return steps                 # n * n -> O(n^2)

def triangle(n):                 # inner loop depends on i
    steps = 0
    for i in range(n):
        for j in range(i):
            steps += 1
    return steps                 # 0+1+...+(n-1) = n(n-1)/2 -> O(n^2)

n = 8
print("sequential, n=8 -> steps =", sequential(n))
print("nested,     n=8 -> steps =", nested(n))
print("triangle,   n=8 -> steps =", triangle(n))

## 5. Feel the growth: n vs n² vs 2ⁿ (no timing needed)

No stopwatch, no loops — just arithmetic. The table prints how many steps each shape needs for small n.

**What to observe / predicted output:** watch the last column explode. It is the rumour that doubles every hour — each +1 to n **doubles** 2ⁿ.

```
   n |      n |       n^2 |        2^n
   1 |      1 |         1 |          2
   2 |      2 |         4 |          4
   4 |      4 |        16 |         16
   8 |      8 |        64 |        256
  16 |     16 |       256 |      65536
  24 |     24 |       576 |   16777216
  30 |     30 |       900 | 1073741824
```

At n = 30, O(n) is 30 chai cups, O(n²) is 900 handshakes, and O(2ⁿ) has already crossed **one billion**. This is why an exponential solution dies before n reaches even 40.

In [ ]:
print(f"{'n':>4} | {'n':>6} | {'n^2':>9} | {'2^n':>10}")
for n in [1, 2, 4, 8, 16, 24, 30]:
    print(f"{n:>4} | {n:>6} | {n*n:>9} | {2**n:>10}")

## 6. My self-audit — today's real task

Now I open my own solutions (Day-1 digit problems + patterns 1–15) and derive each one's complexity myself. No copying answers.

### The checklist (run it on every problem)

1. **Find the loops.** No loop → almost certainly O(1).
2. **How many times does the loop body run as n grows?** Count, don't guess. Key trap: a loop doing `num //= 10` consumes one **digit** per pass, so it runs O(number of digits) = O(log₁₀ n) times — not O(n).
3. **How do the loops combine?** One after another → **add**. One inside another → **multiply**. Inner loop bound depends on `i` → triangle sum 1+2+...+n = n(n+1)/2 → O(n²).
4. **Simplify.** Drop constants, drop smaller terms.
5. **Space:** what did I *create* that grows with the input? Only loop counters and a few variables → O(1). A new list/string of size n → O(n). Printed output does not count.
6. **Write one line of "why".** If I can't write the why, I don't know it yet.

### Type-level hints (I still verify each of mine)

- Digit-loop problems run once per digit → **O(log₁₀ n)** time.
- A full pattern grid prints about n² characters (spaces included) → **O(n²)** time, usually **O(1)** extra space.
- Sum of first N — careful: does its loop run per digit or per value? And is there an O(1) formula?

### My audit table (fill in)

| Problem | Time | Space | Why |
|---|---|---|---|
| Sum of first N | | | |
| Reverse a number | | | |
| Count digits | | | |
| Palindrome number | | | |
| Armstrong number | | | |
| Pattern 1 | | | |
| Pattern 2 | | | |
| Pattern 3 | | | |
| Pattern 4 | | | |
| Pattern 5 | | | |
| Pattern 6 | | | |
| Pattern 7 | | | |
| Pattern 8 | | | |
| Pattern 9 | | | |
| Pattern 10 | | | |
| Pattern 11 | | | |
| Pattern 12 | | | |
| Pattern 13 | | | |
| Pattern 14 | | | |
| Pattern 15 | | | |